# ZuCo Thought Embedding: ZTE

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victor-iyi/zte/blob/main/notebooks/zte_colab.ipynb)

**Mount → train → run the study suite → explore → back up to Drive.** Train on a powerful Colab Pro GPU, explore the results **inline** (tables, charts, figures), and keep everything **permanently on Drive** so a dropped runtime never loses work — then run inference locally.

- **Platform-adaptable & auto-accelerated.** `--device auto` (the default) picks **CUDA (Colab GPU) → Cloud TPU (torch_xla) → Apple MPS → CPU**. Nothing to configure.
- **Resumable & runtime-loss-proof.** Every long run is `--resume`-safe. Point `OUT_ROOT` at Drive (Sections 6/6b) to persist runs the instant they finish, and call `backup_to_drive()` (Section 4) anytime for a provenance-stamped archive + a browsable mirror.
- **Reproducible.** Each run keeps its exact resolved `config.yaml`; `zte-pack` archives carry a `PROVENANCE.json`/`PROVENANCE.md` (git commit + package versions + per-run metrics) so any result can be reproduced and trusted.
- **Single fixed seed (42)** by default for clean, comparable runs; bump to multiple seeds where you want confidence intervals.
- **Encoder *and* decoder.** Sections 5–7 train and judge the EEG encoder; **Section 8** puts a frozen `Qwen2.5-0.5B` behind it on a 227k-parameter prefix bridge and reads text out — with the five brain-independent controls and the verdict gate that decide whether the text means anything.
- **No Colab surprises.** Section 2 sets the env vars Colab leaves unset and fixes the working directory / output paths so the CLIs never error on a fresh runtime.

> ZTE requires **Python 3.14** (Colab ships an older Python), so we use [`uv`](https://docs.astral.sh/uv/) to provision it — one cell, no system changes. All ZTE code therefore runs via `!uv run …` (the 3.14 venv); the plain notebook kernel is used only to read result files for the inline exploration in Section 9b.

> **How to read every number here.** The result is `scoreboard.held_out_retrieval` — the one subject the model
> never saw. Pooled `sentence_retrieval` also scores the 11 training brains, so it rewards memorising the cohort
> rather than reaching a stranger, and it is not evidence of generalisation. On the ZAB fold the raw conformer
> lands **32 hits @ Top-5 of 700 (p ≈ 7e-16)**; every band-power arm lands at chance and lives in
> `experiments/archive/`, so the suite runs the raw conformer only. A low subject probe is not disentanglement on
> its own — read it beside the effective-rank ratio, since a collapsed space has nothing left to identify anyone by.

**Pick a GPU runtime now:** `Runtime → Change runtime type → T4/A100 GPU` (or `TPU`).

**What is new in this revision.** The decoder is rebuilt around two mechanisms aimed squarely at the measured
arithmetic — sentence identity needs 9.4512 bits, word count gives away 5.1422 of them free, and the encoder
carries 1.4965. A **semantic rate ladder** caps the conditioning channel at `stages x log2(codes)` bits *by
construction* and reports how many it actually delivered, with one stage reserved for the word count so the
residual is the part the brain supplied. **Word-synchronous lexical evidence** walks a monotonic pointer across
the reading's words as the LM decodes, re-injecting the brain at every step instead of only through the prefix —
and because the pointer schedule depends on the word count alone, every control inherits it, which is what makes
the new `length_only` control fair. On the encoder side, a **token-level lexical loss** finally asks each word's
EEG to mean that word, with same-word cross-reader positives. Section 6d runs the whole study in one command and
Section 11 turns it into charts.


## 1 · Set up (uv provisions Python 3.14 + installs ZTE)

Re-running this cell **fast-forwards the checkout to the latest `main`** (`git fetch` + `git reset --hard`), so a fix you push from your machine actually reaches the Colab runtime — a persistent runtime otherwise keeps running the old clone. It touches only git-tracked files, so `res/cache/` and everything on Drive are left alone and **no data is re-processed**.

In [ ]:
%%bash
pip install -q uv
# Get into the repo dir whether this is a fresh runtime (/content) or a re-run already inside zte/.
if [ -f pyproject.toml ]; then :
elif [ -d zte/.git ]; then cd zte
else git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main && cd zte
fi
# Fast-forward to the latest pushed main; only git-tracked files move, so res/cache and Drive are untouched.
git fetch --depth 1 origin main && git reset --hard FETCH_HEAD
echo "ZTE @ $(git rev-parse --short HEAD): $(git log -1 --pretty=%s)"
# Provision Python 3.14 + install torch (CUDA wheel on a Colab GPU) and all extras. Cached across runs.
uv python install 3.14
uv sync --all-groups

## 2 · Bootstrap the environment

Colab does not set the env vars headless plotting / tokenizers expect, and the CLIs use paths relative to the repo root. This cell sets those vars for every `!uv run …` subprocess and creates the `res/` output directories, so nothing errors later. Idempotent — safe to re-run.

> **HuggingFace token (optional but recommended).** To silence the *“You are sending unauthenticated requests to the HF Hub”* warning and get higher rate limits + faster downloads for the frozen text encoder (E5, Section 5) and the frozen decoder LM (`Qwen2.5-0.5B`, Section 8), add your token as a Colab secret: **left sidebar → 🔑 → add `HF_TOKEN`** (value from <https://huggingface.co/settings/tokens>) and toggle **notebook access** on. This cell reads it via `google.colab.userdata` and exports `HF_TOKEN` to every subprocess. Without it, downloads still work — just unauthenticated.

In [ ]:
import os

# Enter the repo in the notebook kernel so relative paths and every `!uv run` subprocess resolve;
# this persists across cells, which a %%bash `cd` cannot.
if os.path.isdir('zte') and not os.path.isfile('pyproject.toml'):
    os.chdir('zte')

# Set in the notebook kernel so every `!uv run` subprocess inherits them.
# MPLBACKEND is FORCED: Colab sets it to an inline backend that crashes a headless subprocess.
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

# Stream ZTE's logs LIVE. Python block-buffers stdout when it is not a terminal, and a `!uv run`
# subprocess never is -- so without this the whole run looks silent, then dumps every line at once.
os.environ['PYTHONUNBUFFERED'] = '1'

# Let the CUDA allocator grow its segments instead of fragmenting; a raw batch allocates a few very
# large activation blocks, which is exactly the pattern that strands free memory without this.
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('MPLCONFIGDIR', os.path.abspath('res/.cache/matplotlib'))

# Export the HF_TOKEN Colab secret to every `!uv run` subprocess for authenticated Hub downloads.
# A no-op off Colab or without the secret; downloads then fall back to rate-limited requests.
try:
    from google.colab import userdata  # type: ignore[import-untyped]

    _hf_token = userdata.get('HF_TOKEN')
except Exception as exc:  # not on Colab, or the secret is missing / not granted to this notebook
    _hf_token = None
    print(f'HF_TOKEN unavailable ({type(exc).__name__}) — HuggingFace downloads will be unauthenticated.')
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets — authenticated HuggingFace Hub downloads enabled.')

# Create res/ dirs + report the resolved root/accelerator via the 3.14 venv.
!uv run python -c "from zte.utils import bootstrap; import json; print(json.dumps(bootstrap(chdir=True, quiet=True), default=str, indent=1))"

## 3 · Confirm the accelerator — and how ZTE adapts to it
`--device auto` (the default in every cell) picks the best backend — **CUDA → Cloud TPU → Apple MPS → CPU** — and ZTE then tunes itself to that hardware **automatically, without touching accuracy**:

- **A100 / H100 (CUDA, Ampere+):** bf16 mixed precision + **TF32** matmuls (a large, free speedup; fp32 master weights keep accuracy). Older CUDA falls back to fp16 + GradScaler.
- **Cloud TPU (v6e etc.):** bf16 (TPUs are bf16-native) + **static-shape padding** so XLA compiles once instead of recompiling per batch — padded positions are masked out, so results are unchanged.
- **Apple Silicon (MPS):** stable fp32 (MPS autocast is still maturing) — fully GPU-accelerated locally, and the flagship now runs end-to-end here (the `pdist` op was replaced with a portable equivalent).
- **CPU:** fp32, single-process loading.

DataLoader workers are auto-picked per backend too. The cell below prints both what was **detected** and exactly what **ZTE will use**. Everything is overridable via `--precision`, `--num-workers`, `--compile`, `--static-shapes` on `zte-run`/`zte-benchmark`.

In [ ]:
%%bash
uv run python - <<'PY'
import json

from zte.device import auto_num_workers, resolve_device
from zte.utils.env import accelerator_info

info = accelerator_info()
spec = resolve_device('auto')  # what ZTE will use with --device auto
plan = {
    'backend': spec.kind,
    'device': spec.name,
    'autocast_dtype': str(spec.autocast_dtype).replace('torch.', '') if spec.autocast_dtype else 'fp32',
    'mixed_precision': spec.use_amp,      # bf16 on Ampere+/TPU, fp16 on older CUDA, off on MPS/CPU
    'pin_memory': spec.supports_pin_memory,
    'dataloader_workers_auto': auto_num_workers(spec, -1),
    'tf32_matmul': spec.kind == 'cuda',   # Ampere+ (A100/H100): free matmul speedup
    'static_shapes': spec.kind == 'xla',  # TPU only: fixed-length padding (accuracy-neutral)
}
print(json.dumps({'detected': info, 'zte_will_use': plan}, indent=2))
PY

### (Optional) Cloud TPU
On a **TPU** runtime, install `torch_xla` so `--device auto` selects it. `torch_xla` must match the installed torch; this is best-effort (GPU is the primary, tested path). Uncomment to try:

In [ ]:
# !uv pip install -q torch_xla   # then re-run the accelerator cell above; it should report a Cloud TPU

## 4 · Get data + set up permanent Drive backup
**A) Synthetic (default, no dataset).** A fabricated ZuCo tree — validates the whole pipeline in minutes. The `--synthetic` cells below use it automatically; skip this cell if that is all you want.

**B) Real ZuCo + permanent backup.** Mount Drive, read the dataset **directly** from `Sharables/ZTE/ZuCo Dataset` (mounting is faster than re-downloading), and set up the backup target. Everything this session produces is saved under a **date-stamped** folder `Sharables/ZTE/{RUN_DATE}/` — so a dropped Colab runtime never loses finished work. The cell defines `backup_to_drive()`, used throughout. Shareable ZTE folder: <https://drive.google.com/drive/folders/13EYW1h6dHD5E4YoEWNsKe6ZBHmMU_kFQ>.

In [ ]:
from google.colab import drive  # type: ignore[import-untyped]

drive.mount('/gdrive')

import datetime
import glob
import json
import os
import pathlib
import shutil
import subprocess

# --- One shareable ZTE folder holds everything (data + every session's outputs) ---
# https://drive.google.com/drive/folders/13EYW1h6dHD5E4YoEWNsKe6ZBHmMU_kFQ
ZTE_DRIVE = '/gdrive/My Drive/Sharables/ZTE'
DATA_DIR = f'{ZTE_DRIVE}/ZuCo Dataset'  # the ZuCo .mat files on your Drive
# To RESUME an interrupted session, set RESUME_DATE to its date (e.g. '2026-07-12'); else None = today.
RESUME_DATE = None
RUN_DATE = RESUME_DATE or datetime.date.today().isoformat()  # groups this session's outputs on Drive
DRIVE_DIR = f'{ZTE_DRIVE}/{RUN_DATE}'  # everything this session produces backs up here
for sub in ('', '/experiments', '/archives'):
    os.makedirs(f'{DRIVE_DIR}{sub}', exist_ok=True)
# Expose paths to %%bash cells (which can't read Python vars) as $DATA_DIR / $DRIVE_DIR / ...
os.environ.update(ZTE_DRIVE=ZTE_DRIVE, DATA_DIR=DATA_DIR, DRIVE_DIR=DRIVE_DIR, RUN_DATE=RUN_DATE)

print('data found :', os.path.isdir(DATA_DIR))
print('backups -> :', DRIVE_DIR)
!ls "{DATA_DIR}" | head


RUNS_DIR = f'{DRIVE_DIR}/experiments'  # spotlight cells write here; the suite mirrors here
LOCAL_RUNS = 'res/experiments'  # the suite trains here first (fast disk), then mirrors
os.environ.update(RUNS_DIR=RUNS_DIR)


def run_dirs(*extra: str) -> list[pathlib.Path]:
    """Every run directory this notebook can see — Drive first, then the local disk, deduped by name.

    Runs land in different places depending on the cell that produced them (Section 5 writes straight
    to Drive, Section 6b trains locally then mirrors), and a fresh runtime has an empty local disk.
    """
    seen, out = set(), []
    for base in (RUNS_DIR, *extra, LOCAL_RUNS):
        for mf in sorted(glob.glob(f'{base}/*/manifest.json')):
            d = pathlib.Path(mf).parent
            if d.name not in seen:
                seen.add(d.name)
                out.append(d)
    return out


def show_resources() -> None:
    """Prints RAM/GPU/disk, so an OOM is predictable rather than a mystery kill."""
    import shutil as _sh

    gb = 1 << 30
    total = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / gb
    free = _sh.disk_usage('.').free / gb
    print(f'System RAM: {total:5.1f} GB   |   free disk: {free:5.1f} GB')
    try:
        import torch

        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            print(f'GPU:        {props.name} ({props.total_memory / gb:.1f} GB)')
        else:
            print('GPU:        none  \u2014 Runtime \u2192 Change runtime type \u2192 GPU')
    except ImportError:
        print('GPU:        torch not importable in the notebook kernel (the uv env has it)')
    if total < 20:
        print(
            '\n\u26a0  Raw-EEG bundles are ~24 GB when materialised. This runtime is too small for the\n'
            '   raw configs unless the bundle is memory-mapped (rebuild it once with zte-prepare).\n'
            '   Prefer Runtime \u2192 Change runtime type \u2192 High-RAM.'
        )


def _real_runs(experiments: str = 'res/experiments') -> list[pathlib.Path]:
    """Non-synthetic run dirs — skips --synthetic smoke runs (missing flag = treated as real)."""
    keep = []
    for mf in glob.glob(f'{experiments}/*/manifest.json'):
        try:
            synthetic = json.load(open(mf)).get('synthetic', False)
        except Exception:
            synthetic = False
        if not synthetic:
            keep.append(pathlib.Path(mf).parent)
    return sorted(keep)


def backup_to_drive(note: str | None = None, include_synthetic: bool = False) -> None:
    """Back up REAL (non-synthetic) runs to Drive. Safe to call anytime / repeatedly.

    Smoke / --synthetic runs are skipped by default (pass include_synthetic=True to force); if there
    are no real runs this is a friendly no-op, so nothing pollutes Drive. Produces a restorable,
    provenance-stamped best-only zip under {DRIVE_DIR}/archives/ and a browsable mirror of the real
    runs (reports, figures, 3-D explorers, best.pt) under {DRIVE_DIR}/experiments/. Real benchmarks
    are written straight to Drive by the benchmark cell (Section 7).
    """
    runs = None if include_synthetic else _real_runs()
    if runs is not None and not runs:
        print('No real (non-synthetic) runs to back up — skipping Drive. Pass include_synthetic=True to force.')
        return
    ts = datetime.datetime.now().strftime('%H%M%S')
    arch = f'{DRIVE_DIR}/archives/zte_{RUN_DATE}_{ts}.zip'
    cmd = ['uv', 'run', 'zte-pack', 'zip', '--all', '--best-only', '--out', arch]
    if not include_synthetic:
        cmd.append('--skip-synthetic')
    if note:
        cmd += ['--note', note]
    print('-> provenance zip:', arch)
    subprocess.run(cmd, check=False)
    ignore = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')
    dst = pathlib.Path(DRIVE_DIR) / 'experiments'
    if runs is None:
        src = pathlib.Path('res/experiments')
        if src.is_dir():
            shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    else:
        for r in runs:
            shutil.copytree(r, dst / r.name, dirs_exist_ok=True, ignore=ignore)
    print(f'backed up {"all" if runs is None else len(runs)} run(s) to', DRIVE_DIR)


def snapshot_to_drive(
    note: str | None = None,
    targets: list[str] | None = None,
    move: bool = False,
    include_synthetic: bool = False,
) -> str:
    """Zip the FULL working state to Drive so you can continue LOCALLY without GPU time.

    Captures res/experiments + res/cache + res/benchmark + res/explorer in ONE provenance-stamped
    zip (the dataset cache means a local session never re-prepares data). --synthetic experiment
    runs are excluded by default (include_synthetic=True to keep them). Download the single file,
    then `zte-pack unpack <zip> --dest res` on your machine to keep exploring / training offline.
    """
    ts = datetime.datetime.now().strftime('%H%M%S')
    out = f'{DRIVE_DIR}/archives/zte_snapshot_{RUN_DATE}_{ts}.zip'
    cmd = ['uv', 'run', 'zte-pack', 'snapshot', *(targets or []), '--out', out]
    if not include_synthetic:
        cmd.append('--skip-synthetic')
    if note:
        cmd += ['--note', note]
    if move:
        cmd.append('--move')
    print('-> full snapshot ->', out)
    subprocess.run(cmd, check=False)
    print('snapshot on Drive:', out)
    return out


def remove_from_res(*names: str) -> None:
    """Easily delete run dirs / res/ subpaths locally to free space (does NOT touch Drive).

    remove_from_res('colab_exp6')                  # a run name under res/experiments/
    remove_from_res('res/benchmark', 'res/cache')   # any res/ subpath
    """
    for n in names:
        p = pathlib.Path(n)
        if not p.exists():
            p = pathlib.Path('res/experiments') / n  # bare run name
        if p.exists():
            shutil.rmtree(p)
            print('removed', p)
        else:
            print('not found:', n)


def restore_from_drive(
    run_date: str | None = None, drive_sub: str = 'experiments', local: str = 'res/experiments'
) -> None:
    """Pull a Drive session's runs back to local so you can resume after a runtime reset.

    Copies {ZTE_DRIVE}/<date>/<drive_sub>/* -> <local>/ (checkpoints, config, eval), then re-run the
    training cell with --resume: finished runs skip, interrupted ones continue from their last checkpoint.
    Defaults to this session's RUN_DATE + the flat experiments/ dir; for the LOSO sweep pass
    restore_from_drive(drive_sub='loso', local='res/experiments/loso').
    """
    date = run_date or RUN_DATE
    src = pathlib.Path(f'{ZTE_DRIVE}/{date}/{drive_sub}')
    if not src.is_dir():
        print('nothing to restore at', src)
        return
    dst = pathlib.Path(local)
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'restored {len(list(src.iterdir()))} item(s) from {src} -> {dst}. Re-run with --resume to continue.')


def mirror_to_drive(local: str, drive_sub: str | None = None) -> None:
    """Copy a local dir to Drive (browsable), minus heavy transient files (cache/tb/bundle/last.pt/epoch ckpts).

    Persists FULL runs (eval, figures, interactive, COMPARE.html) after training locally with DRIVE_BACKUP.
    e.g. mirror_to_drive('res/experiments/loso', 'loso').
    """
    src = pathlib.Path(local)
    if not src.is_dir():
        print('nothing to mirror at', local)
        return
    dst = pathlib.Path(DRIVE_DIR) / (drive_sub or src.name)
    ignore = shutil.ignore_patterns('cache', 'tb', 'bundle', 'ckpt_epoch*.pt', 'last.pt')
    shutil.copytree(src, dst, dirs_exist_ok=True, ignore=ignore)
    print(f'mirrored {src} -> {dst}')

## 4b · Process the dataset ONCE — on Drive, then never again

The slow part of every run is turning the raw ZuCo `.mat` files into tensors (band power, imputation,
normalisation). That result depends **only on the dataset config**, not on the model or objective, so it is
identical across experiments, held-out subjects, seeds and sessions.

The cache is **layered and two-level**, so this is paid once for the whole project:

- **Layered** — every command reads a fast **local** copy first, then the persistent **Drive** copy
  (`Sharables/ZTE/prepared/`). A Drive hit is copied down once, on demand; a freshly built bundle is pushed to
  Drive **immediately** (not at the end of a run), so a reclaimed VM never re-processes. Set once via
  `ZTE_CACHE_REMOTE` and every `zte-*` command uses it — no `--data-cache` flag needed.
- **Two-level** — the expensive `.mat` *extraction* is cached separately from the cheap *processing*. A config
  that only changes normalisation, imputation, eye-tracking or length filters **reuses the extraction and
  re-derives in seconds** instead of re-parsing every `.mat` file.
- **Artifacts too** — the frozen encoder passes (the contextual BERT meaning matrix, the E5 sentence
  embeddings) are content-addressed and layered onto the same Drive store, so they are built once ever
  rather than once per runtime.

> **Nothing unzips on a warm session.** Your ZuCo folder on Drive holds the task `.zip` archives, so
> "resolving the data source" means unpacking tens of gigabytes onto the VM. Every command now asks the
> bundle store *first* and skips resolving the source entirely when the processed dataset already exists —
> so a warm run never touches the archives. You'll see `Processed bundle already persistent; skipping
> raw-data extraction.` in the log.

> **This cell is a Drive lookup, not a build.** `zte-prepare --configs` keys every config first, checks
> whether the persistent store already holds each dataset, and **only then** touches the raw data — and only
> for what is genuinely missing. Nothing is rebuilt, and nothing is even copied down, when Drive already has
> it. On a fully-prepared project it is a metadata check that finishes in under a second, so re-running it on
> every fresh runtime costs nothing.
>
> There is deliberately **no "already done" sentinel**. The old one lived on the VM's local disk, which Colab
> wipes between sessions, so it never fired where it mattered — and worse, a sentinel goes stale the moment you
> add a new experiment config, silently skipping the dataset that config needs. Asking the store is both
> cheaper and correct.

Read the `status` column: `cached` = already on this VM, `on-drive` = in the persistent store and pulled down
by the first run that needs it, `MISSING` = about to be built. Pass `--check` to report without building.

In [ ]:
# Point every zte-* command at the persistent Drive store, then confirm the datasets exist there.
import os

PREPARED_DRIVE = f'{ZTE_DRIVE}/prepared'  # persistent, NOT date-stamped — reusable forever
PREPARED_LOCAL: str = 'res/cache/prepared'  # fast local copy runs read from (wiped with the runtime)

# One environment variable wires the persistent store into EVERY zte-* command (run, prepare, benchmark,
# the sweep scripts). No per-cell --data-cache flag needed; it also still honours one if you pass it.
os.environ['ZTE_CACHE_REMOTE'] = PREPARED_DRIVE
os.environ['DATA_CACHE'] = PREPARED_LOCAL  # the sweep scripts pass this as --data-cache (local read copy)

# Cheap and idempotent: this keys every shipped config, asks the store what it already holds, and builds only
# what is genuinely absent. A fully-prepared project never touches the raw .mat files at all.
!uv run zte-prepare --root "{DATA_DIR}" --configs --cache-dir "{PREPARED_LOCAL}" --cache-remote "{PREPARED_DRIVE}"


def _sync_bundles(src: str, dst: str, label: str) -> None:
    """Copy content-addressed cache entries src->dst, skipping any already present (they are immutable)."""
    s = pathlib.Path(src)
    if not s.is_dir():
        print(f'no processed-dataset cache at {src} yet — {label}')
        return

    d = pathlib.Path(dst)
    d.mkdir(parents=True, exist_ok=True)
    n = 0
    for sub in s.iterdir():
        if sub.is_dir() and not (d / sub.name).exists():
            shutil.copytree(sub, d / sub.name)
            n += 1
    print(f'{label}: {n} new entry(ies)  {src} -> {dst}')


def restore_prepared_cache() -> None:
    """Pull every Drive bundle down at once (per-command staging is automatic; this just front-loads it)."""
    _sync_bundles(PREPARED_DRIVE, PREPARED_LOCAL, 'restored')


def backup_prepared_cache() -> None:
    """Push newly-built bundles local -> Drive; per-command publishing already does this automatically."""
    _sync_bundles(PREPARED_LOCAL, PREPARED_DRIVE, 'backed up')

## 5 · Run the SOTA experiments — one spotlight run, or a resumable series

**`experiments/flagship/zte_raw_aligned.yaml` (exp12) is the champion candidate.** It is the measured raw-conformer
arm (32 hits @ Top-5 of 700, p ≈ 7e-16) with its encoder left byte-for-byte identical, plus three label-free steps
that close the gap the 2026-07-25 re-scoring exposed.

**The gap.** The raw path had never had cross-subject alignment of *any* kind. `dataset.normalize` only ever applied
to band power, so `normalize: riemannian` was a silent no-op for every raw run on the board — the winning arm was
training on unaligned voltages, and its subject probe was still 0.41 against a 0.81 raw-feature baseline. Identity
was never removed; it was simply never addressed on this path.

**The three changes**, each separately ablatable in 5c:

1. **Euclidean alignment** (`dataset.raw_align: euclidean`) — whiten every subject by the inverse square root of
   their own mean channel covariance (He & Wu 2019), so all brains arrive with the same second-order statistics.
   Fitted on `all` subjects, held-out included: it reads no label, no text and no split, only that person's own
   voltages. That is the calibration a real BCI runs while fitting the cap, not a peek at the answers.
2. **A subject adapter driven by inferred statistics, not an ID lookup** (`model.subject_adapter`). This is the
   novel piece. The standard trick (Défossez 2023) is a per-subject layer indexed by subject ID — which cannot help
   the one subject that matters in LOSO, because an unseen ID has no learned row and a zero-init table makes the
   held-out person the *identity map*. The adapter instead reads that person's covariance-geometry **signature** and
   a hypernetwork emits their adapter weights from it, so subject 13 is an interpolation in signature space rather
   than a missing row. The closest published work states outright that it cannot generalise to unseen subjects.
3. **Identity orthogonality** (`objective.identity_orthogonality_weight`) — remove identity by *decorrelating*
   content from the signature rather than by making identity unpredictable. A gradient-reversal adversary is
   satisfiable by collapse, which is exactly how band power reached eff-rank 0.160; this term costs a full-rank
   identity-free space nothing. The adversary is halved to 0.05 so the two are not fighting the same gradient.

**Reading the result.** The scoreboard now leads with **rank percentile + a 95% bootstrap CI**, because every query
contributes to it, and reports Top-K as **raw hit counts against the handful expected by chance with an exact
binomial tail**. Top-1 on 700 queries at 1/700 expects *one* hit — "0.006 vs 0.001" is three hits and noise.

**The arms:**
- `zte_raw_aligned.yaml` — exp12, the champion candidate (exp10's encoder + the alignment stack).
- `archive/zte_raw_aligned_wide.yaml` — retired 2026-08-14: rank percentile 0.9523 (0.9479–0.9565), below the retained set with no interval overlap. Its v2 encoder (64 filters, multiscale bank, attentive pool)
  alone reached the healthiest geometry on the board (eff-rank 0.535) but the weakest Top-1 — a bigger model
  underfitting. Does that capacity convert once alignment stops making it pay for identity?
- `clip_e5_meaning_raw.yaml` — exp10, **the measured baseline to beat** (32/700 @ Top-5).
- `clip_e5_raw.yaml` — exp8, the same without meaning distillation. Equal retrieval, worse subject probe.

**Turn-key ingredients (no manual prereq cells).** `--spatial {exact,attention,approx,off}` builds + wires the
montage; `--meaning {static,contextual,hash}` builds + wires the distillation target. Each artifact is built **once
and cached under `res/`** and reused across every held-out subject and ablation arm. Alignment is applied *after*
the cached bundle loads, so **turning it on never invalidates a prepared bundle**.

**How this section is organised.** Everything writes straight to Drive and is `--resume`-safe — a complete run is
skipped *instantly*, an interrupted one continues from its last checkpoint.

- **5-iii** — one spotlight experiment: a single config × one held-out subject. Start here.
- **5-iv** — the flagship series: exp12 against the two raw arms it is tied with.
- **5-v / 5-vi** — the exp12 **one-knob ablations**: which of the three changes actually did it.

For the exhaustive multi-hour runs use **Section 6** (full LOSO over *every* subject) and **6b** (the full suite).

### 5a · One spotlight run — a single config x one held-out subject

Trains, fully evaluates, and writes everything straight to Drive. `--spatial` / `--meaning` build the montage
and meaning target once (cached under `res/`, reused by every later run), and `--data-cache` reuses the
processed bundle, so the `.mat` load and processing are skipped entirely. `--resume` is idempotent: a finished
run is skipped instantly, an interrupted one continues.

`CONFIG` picks the arm. The three retained flagships are tied on the held-out board, so any is a
defensible default: `zte_raw_aligned.yaml` (exp12, rank percentile 0.9672), `clip_e5_meaning_raw.yaml`
(0.9667), `clip_e5_raw.yaml` (0.9635, and the only one whose length-stratified Top-1 clears *p* < 0.05)
(32 hits @ Top-5 of 700).

To sanity-check the mechanics with no data at all, swap the flags for `--synthetic --epochs 3 --out-root
res/experiments`. **A synthetic run is never a result.**

In [ ]:
# The new flagship: exp12's alignment stack plus the token-level lexical loss. Its per-word projection is
# what the v2 decoder's evidence path reads, so train THIS one before Section 8.
CONFIG: str = 'experiments/flagship/zte_lexical_raw.yaml'  # or flagship/zte_raw_aligned.yaml (the lexical-off pair)
HOLDOUT: str = 'ZAB'  # the held-out 'new brain' for this run
SPATIAL: str = 'exact'  # exact ZuCo-105 montage (spatial encoding + regions); or approx / attention / off / keep
MEANING: str = 'keep'  # vocab-restricted GloVe target; or contextual (per-occurrence BERT mid-layer) / hash / keep

!uv run zte-run --config {CONFIG} --root "{DATA_DIR}" --spatial {SPATIAL} --meaning {MEANING} \
    --data-cache "{PREPARED_LOCAL}" --loso-holdout {HOLDOUT} --out-root "{DRIVE_DIR}/experiments" --resume
backup_prepared_cache()  # persist the processed bundle to Drive so future sessions skip re-processing

# Resolve the run dir from the config's own run_name + the LOSO suffix, then show the honest headline:
import yaml

_run_name = yaml.safe_load(open(CONFIG))['run_name']
RUN_DIR = f'{DRIVE_DIR}/experiments/{_run_name}_lo{HOLDOUT}'
rep = pathlib.Path(f'{RUN_DIR}/evaluation/report.md')
if rep.exists():
    t = rep.read_text()
    print(t[t.find('## Scoreboard') : t.find('## Verdict')] or t[:1500])
print('run dir ->', RUN_DIR)
print(
    'interactive held-out scoreboard ->',
    f'{RUN_DIR}/evaluation/interactive/held_out_scoreboard.html',
)

### 5b · A series of spotlight runs

Each `(config, held-out subject)` pair is trained, fully evaluated and written straight to Drive. `--resume`
skips any run already complete and continues an interrupted one, so re-running this cell never redoes
finished work — leave finished entries in the list.

The montage, meaning target and processed dataset are built by the first run and reused from cache by the rest.

In [ ]:
SPATIAL: str = 'exact'  # exact ZuCo-105 montage for every run; or approx / attention / off / keep
MEANING: str = 'keep'  # vocab-restricted GloVe target; or contextual (per-occurrence BERT) / hash / keep
SPOTLIGHT = [
    (
        'experiments/flagship/zte_raw_aligned.yaml',
        'ZAB',
    ),  # exp12: the alignment stack on exp10's encoder — the champion candidate
    (
        'experiments/flagship/clip_e5_meaning_raw.yaml',
        'ZAB',
    ),  # exp10: the MEASURED baseline to beat — 32 hits @ Top-5 of 700, p ~ 7e-16
    (
        'experiments/flagship/clip_e5_raw.yaml',
        'ZAB',
    ),  # exp8: the same without meaning distillation (equal retrieval, worse subject probe)
]

import pathlib

import yaml

# Montage, meaning target AND the processed dataset are built once by the first run and reused from cache.
for _cfg, _holdout in SPOTLIGHT:
    _rn = yaml.safe_load(open(_cfg))['run_name']
    _dir = f'{DRIVE_DIR}/experiments/{_rn}_lo{_holdout}'
    _done = pathlib.Path(f'{_dir}/evaluation/report.md').exists()
    print(
        f'=== {_rn} · held-out {_holdout} · {"already complete (skipped)" if _done else "training/resuming"} -> {_dir}'
    )
    !uv run zte-run --config {_cfg} --root "{DATA_DIR}" --spatial {SPATIAL} --meaning {MEANING} --data-cache "{PREPARED_LOCAL}" --loso-holdout {_holdout} --out-root "{DRIVE_DIR}/experiments" --resume
backup_prepared_cache()  # persist the processed bundle(s) to Drive for future sessions
print('\ndone — spotlight runs are on Drive under', f'{DRIVE_DIR}/experiments')
print('Any run marked ✗ above did NOT train; nothing downstream will show results for it.')

### 5c · Which of the three changes actually did it — the exp12 ablations

A stack of three changes that wins tells you nothing about *which* change won, and the honest board has already
burned us once for accepting a headline without checking what produced it. Each config below is **byte-identical to
`zte_raw_aligned.yaml` except for one knob**, so any difference is attributable rather than asserted.

- **`exp12_align_off`** — Euclidean alignment off, adapter and orthogonality on. If the adapter can infer and cancel
  the individual forward model by itself, this loses little; if whitening is doing the work, it falls back toward
  the 32/700 baseline.
- **`exp12_adapter_off`** — alignment on, hypernetwork off. Whitening is linear and one-size; this measures how much
  of the gap is *nonlinear residual* that only a conditioned encoder can reach.
- **`exp12_orthogonality_off`** — the rank-preserving identity penalty off, adversary restored to the baseline 0.1.
  **Watch effective rank here, not just retrieval** — that is the failure mode this term exists to prevent.
- **`exp12_align_fit_train`** — alignment fitted on training subjects only, so the held-out subject falls back to
  the cohort reference. The strict ablation for anyone who doubts that label-free calibration on the held-out
  subject is legitimate: it shows what the number would be if a new user were never allowed to calibrate their cap.

The text-encoder A/B (E5 vs Qwen vs BGE vs MPNet) that used to live here is **retired** — every arm scored p ≥ 0.07
on the held-out board and all were band-power. They are in `experiments/archive/` with their numbers.

Each config below differs from `experiments/flagship/zte_raw_aligned.yaml` by exactly **one** knob, held out on ZAB. Same `--resume` semantics as 5b.

In [ ]:
SPATIAL: str = 'exact'  # exact ZuCo-105 montage for every arm; or approx / attention / off / keep
ABLATIONS = [
    (
        'experiments/ablation/exp12_align_off.yaml',
        'ZAB',
    ),  # Euclidean alignment OFF (adapter + orthogonality still on)
    (
        'experiments/ablation/exp12_adapter_off.yaml',
        'ZAB',
    ),  # subject hypernetwork OFF (alignment + orthogonality still on)
    (
        'experiments/ablation/exp12_orthogonality_off.yaml',
        'ZAB',
    ),  # identity-orthogonality OFF, adversary back to 0.1 — watch effective rank
    (
        'experiments/ablation/exp12_align_fit_train.yaml',
        'ZAB',
    ),  # alignment fitted on TRAIN subjects only — the no-calibration ablation
]

import pathlib

import yaml

for _cfg, _holdout in ABLATIONS:
    _rn = yaml.safe_load(open(_cfg))['run_name']
    _dir = f'{DRIVE_DIR}/experiments/{_rn}_lo{_holdout}'
    _done = pathlib.Path(f'{_dir}/evaluation/report.md').exists()
    print(f'=== {_rn} · held-out {_holdout} · {"already complete (skipped)" if _done else "training/resuming"}')
    !uv run zte-run --config {_cfg} --root "{DATA_DIR}" --spatial {SPATIAL} --data-cache "{PREPARED_LOCAL}" --loso-holdout {_holdout} --out-root "{DRIVE_DIR}/experiments" --resume
backup_prepared_cache()  # persist the processed bundle(s) to Drive for future sessions
print('\ndone — compare each ablation against exp12_zte_raw_aligned to see what earned the win.')

### 5d · Compare the spotlight runs

Builds the cross-run dashboard from Drive and renders it inline: cross-subject retrieval, rank percentile and
effective rank on the held-out north-star, beside the honesty checks.

The pooled `sentence_retrieval` Top-1 is deliberately absent — it is computed over the 11 training brains too,
so it is not evidence that anything reached a stranger.

In [ ]:
!uv run zte-compare --experiments "{DRIVE_DIR}/experiments" --out "{DRIVE_DIR}/experiments/COMPARE.html"

# A compact headline table across the runs (reads each run's metrics.json — no ZTE import needed):
import pandas as pd
from IPython.display import HTML, display  # type: ignore[import-untyped]

_rows = []
for _mf in sorted(pathlib.Path(f'{DRIVE_DIR}/experiments').glob('*/evaluation/metrics.json')):
    _m = json.load(open(_mf))
    _sb = (_m.get('scoreboard') or {}).get('held_out_retrieval') or {}
    _n = _sb.get('n_queries') or 0
    _rows.append(
        {
            'run': _mf.parent.parent.name,
            'held_out_top1': _sb.get('top1'),
            'held_out_lift_top1': _sb.get('lift_top1'),
            'rank_percentile': _sb.get('rank_percentile'),
            'top5_hits': round((_sb.get('top5') or 0) * _n),
            'top5_p': _sb.get('top5_p'),
            'n_queries': _n,
            'eff_rank_ratio': (_m.get('embedding_health') or {}).get('effective_rank_ratio'),
            'perm_above_chance': (_m.get('verdict') or {}).get('retrieval_above_chance_perm'),
        }
    )
if _rows:
    display(pd.DataFrame(_rows).set_index('run'))
display(HTML(open(f'{DRIVE_DIR}/experiments/COMPARE.html', encoding='utf-8').read()))

## 6 · Full LOSO on the SOTA config — the “new brain” sweep (multi-hour, resumable)

Once a spotlight run (Section 5) looks good, rotate the held-out subject over the **whole cohort** with the same config, turning one data point into a **trend** (`COMPARE.html`). `FULL_CFG` selects it (defaults to the SOTA config). This is the exhaustive, **multi-hour** away-game — one model per held-out subject. Fast local training + a live per-epoch checkpoint mirror to Drive; every run is `--resume`-safe (finished subjects skip instantly, an interrupted one continues), so a dropped runtime never loses work — just re-run. The **real** sweep runs by default below; the synthetic dry-run is commented out.

One config x every subject: the exhaustive "does it generalise to any stranger" trend. **Multi-hour.**
Training runs on the fast local disk with a live per-epoch checkpoint mirror to Drive, and the built-in
`--resume` skips finished subjects, so the cell is safe to re-run after a dropped runtime.

`SPATIAL` / `MEANING` / `DATA_CACHE` provision the montage, meaning target and processed dataset once and reuse
them across all 12 folds — all three are subject-independent, so nothing is recomputed per subject.
`FULL_CFG` selects the arm: `zte_raw_aligned` (exp12) by default, or either flagship tied with it
or `clip_e5_meaning_raw.yaml` (the 32-of-700 baseline) once their spotlight looks good.

On completion `run_loso.sh` writes `LOSO_SUMMARY.md` — the honest held-out headline, leading with rank
percentile and a bootstrap CI and reporting Top-K as hit counts with an exact binomial tail — plus
`COMPARE.html`. Top-1 on 700 queries at 1/700 chance expects **one** hit, so rates there are not readable.

Other knobs: `SUBJECTS="ZAB ZDM"` restricts the held-out set, `OUT_ROOT` writes straight to Drive, and
`SMOKE=1 bash scripts/run_loso.sh` dry-runs the whole sweep on synthetic data, on CPU, in seconds.

In [ ]:
!SPATIAL=exact MEANING=keep DATA_CACHE="{PREPARED_LOCAL}" FULL_CFG=experiments/flagship/zte_raw_aligned.yaml DRIVE_BACKUP="{DRIVE_DIR}/loso" bash scripts/run_loso.sh "{DATA_DIR}"
mirror_to_drive('res/experiments/loso', 'loso')  # push eval/figures/interactive to Drive once training completes
backup_prepared_cache()  # persist the processed bundle to Drive for future sessions

### 6a · The honest LOSO headline

Each fold's `sentence Top-1` in `INDEX.md` is **pooled** over all subjects, so it is dominated by the 11 the model trained on and reads far higher than its true generalisation. `zte-loso-summary` reports the honest number instead: **held-out retrieval** (the never-seen subject alone) as a mean±std lift over chance, how many folds beat chance, the **converged/collapsed** split (the 2026-07-24 seed-42 sweep collapsed on 3/12 folds — rerun with `SEEDS="42 43 44"` to average that out), and the anchor-calibration lift (the most promising signal for the decoder — a new brain snapped into the shared frame from ~12 anchor words).

In [ ]:
# The honest LOSO trend: rank percentile + CI, Top-K as hit counts with an exact binomial tail.
# Reads the local sweep dir if this session produced it, else the Drive mirror from an earlier one.
_local_loso = 'res/experiments/loso'
LOSO_DIR = _local_loso if glob.glob(f'{_local_loso}/*/manifest.json') else f'{DRIVE_DIR}/loso'
print('reading LOSO runs from', LOSO_DIR)
!uv run zte-loso-summary --experiments "{LOSO_DIR}" --out "{LOSO_DIR}/LOSO_SUMMARY.md"
if LOSO_DIR.startswith('res/'):
    mirror_to_drive(LOSO_DIR, 'loso')  # push the summary + csv to Drive alongside the runs

## 6b · Full experiment suite — the fixed-seed driver
`scripts/run_suite.sh` runs the suite at a **single fixed seed (42)**, each arm held out on ZAB. `STUDIES` picks what runs (default `audit flagship ablate`): `audit` = the model-free confound report, `flagship` = the exp12 alignment stack (narrow + wide) against the two measured raw arms it must beat, `ablate` = the four exp12 one-knob studies. `controls` (the skip-gram floor), `benchmark` and `loso` add the honest baseline, the objective sweep and the full 12-subject sweep. **The band-power and text-encoder arms are gone** — every one scored p ≥ 0.03 on the held-out board; they are in `experiments/archive/` with their numbers. Every run is `--resume`-safe, so an interrupted suite continues instantly where it stopped.

**Best of both worlds (the default).** Train on the **fast local disk** while `DRIVE_BACKUP` mirrors each run's `best.pt`/`last.pt` to Drive **every epoch** — so trainable progress is never lost if the runtime dies — and then `backup_to_drive()` writes **everything else** (evaluation, figures, interactive HTML) to Drive once training completes. You get fast training I/O *and* full persistence. Interrupted? Set `RESUME_DATE` (Section 4), `restore_from_drive()`, and re-run: finished runs skip, the interrupted one continues, restored runs re-evaluate (cheap), and the final `backup_to_drive()` re-syncs. The kept-open alternative — `OUT_ROOT=…/experiments` on Drive — writes *everything* to Drive live (simplest resume, no restore) but pays ~20–30 s/epoch on big raw-conformer checkpoints. `SMOKE=1` = fast synthetic dry-run (stays local). Uncomment `study_loso_sweep` inside the script for the full 12-subject leave-one-out sweep on the flagship.

The fixed-seed (42) bias-controlled study set. **Multi-hour.**

Training runs on the fast local disk while `DRIVE_BACKUP` mirrors each run's `best.pt`/`last.pt` to Drive every
epoch, so a dead runtime never costs trainable progress; `backup_to_drive()` then pushes everything else — eval,
figures, interactive pages, benchmark — once training completes. `--resume` is built in.

`SPATIAL=exact` provisions the shared montage once and `DATA_CACHE` reuses the processed bundle across every
run. The suite mixes meaning-distillation and CLIP configs, so `MEANING` is left per-config.

To dry-run the entire suite on synthetic data in seconds: `SMOKE=1 bash scripts/run_suite.sh`. To write
everything straight to Drive as produced (simplest resume, slower checkpoint I/O), swap `DRIVE_BACKUP` for
`OUT_ROOT="{DRIVE_DIR}/experiments"`. After a runtime reset, set `RESUME_DATE` (Section 4), call
`restore_from_drive()` to pull the mirrored checkpoints back, then re-run this cell.

In [ ]:
!SPATIAL=exact DATA_CACHE="{PREPARED_LOCAL}" DRIVE_BACKUP="{DRIVE_DIR}/experiments" BENCH_ROOT="{DRIVE_DIR}/benchmark" bash scripts/run_suite.sh "{DATA_DIR}"
backup_to_drive(note='full suite (seed 42)')  # writes res/experiments/* (eval/figures/interactive) to Drive
backup_prepared_cache()  # persist the processed bundle(s) to Drive for future sessions

## 6c · Prove each lever — single-variable ablation (`zte-ablate`)
The scoreboard only becomes *proof* when each lever is tested in isolation. `zte-ablate generate` writes a config sweep that changes **exactly one knob**; run both arms, then `zte-ablate diff` reports that knob's contribution to the held-out LOSO north-star — everything else identical. This is the discipline the original reports could only apply to VICReg.

`zte-ablate` drives **any** dotted `section.field` with zero code change, so every lever is a clean
single-variable A/B against the held-out LOSO north-star:

| Knob | Values | What it isolates |
| --- | --- | --- |
| `objective.identity_orthogonality_weight` | 0 / 1.0 | the rank-preserving identity penalty (exp12) |
| `dataset.raw_align` | none / euclidean | per-subject whitening of the raw windows (exp12) |
| `model.subject_adapter` | false / true | the signature-driven hypernetwork (exp12) |
| `objective.subject_adversary_weight` | 0 / 0.1 / 0.3 | the adversary orthogonality is meant to replace |
| `objective.all_but_top` | 0 / 1 | the geometry fix (anti-hubness) |
| `objective.csls_neighbors` | 0 / 10 | CSLS retrieval correction |
| `objective.alignment_weight` | 0 / 0.1 | align+uniformity's missing half |
| `objective.tau_plus` | 0 / 0.1 | debiased contrastive |
| `objective.data2vec_aux_weight` | 0 / 0.5 | collapse-insurance / fill nuisance dims |
| `model.spatial_encoding` | spherical_harmonics / spatial_attention | electrode geometry |
| `model.subject_film` | 0 / 1 | FiLM subject conditioning |

For **combinations**, repeat `--knob`/`--values` for the Cartesian product and provision the ingredients once
so every arm shares the exact montage — e.g. spatial encoding x meaning target (3 x 2 = 6 configs) reveals
whether exact geometry only helps once meaning is on:

```sh
uv run zte-ablate generate --config experiments/flagship/zte_raw_aligned.yaml --spatial exact \
    --knob model.spatial_encoding --values none,spherical_harmonics,spatial_attention \
    --knob objective.meaning_contextual --values None,bert-base-uncased --out-dir res/ablate_configs
```

In [ ]:
KNOB = 'objective.identity_orthogonality_weight'  # the exp12 lever; Section 5c runs the pre-built one-knob configs

# 1) generate the one-knob sweep from the SOTA config
!uv run zte-ablate generate --config experiments/flagship/zte_raw_aligned.yaml --knob {KNOB} --values 0,0.3,1.0 --out-dir res/ablate_configs

# 2) run each arm on real data -> Drive. --resume skips arms that are already complete (re-run freely).
for cfg in sorted(glob.glob('res/ablate_configs/*.yaml')):
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --loso-holdout ZAB --out-root "{DRIVE_DIR}/ablate" --resume

# 3) diff the scoreboards -> the knob's isolated contribution on the held-out north-star (retrieval + rank-percentile)
metrics = sorted(glob.glob(f'{DRIVE_DIR}/ablate/*/evaluation/metrics.json'))
if len(metrics) >= 2:
    !uv run zte-ablate diff --knob {KNOB} --baseline {metrics[0]} --variant {metrics[-1]}

## 6d · The whole study in one command — `run_zte_study.sh`

Everything Sections 5-8 do by hand, as one resumable driver: the confound audit, the flagship encoder at
several seeds, the decoder and its one-knob arms over that encoder, the feature-ablation table, the
length-confound audit against every checkpoint, and finally the analysis page. **Multi-hour**, and every step
carries `--resume`, so a reclaimed VM costs at most the epoch in flight — re-run the identical cell and finished
work is skipped instantly.

Everything lands in `DRIVE_DIR/experiments`, so nothing this cell produces lives only on the VM disk.

| Stage | What it answers |
| --- | --- |
| `audit` | Is the dataset confounded, before any model is trained? |
| `encoder` | Does the encoder reach a stranger's brain, at more than one seed? |
| `loso` | …for **every** one of the 12 subjects? (add it explicitly — it is the expensive stage) |
| `decoder` | Does the decoder read the brain, or recite the corpus? |
| `ablation` | Which lever actually did it: raw vs band power, harmonics vs indexing, invariance on vs off? |
| `rebaseline` | How much of the number is sentence length? |
| `analysis` | All of the above, as one offline page plus CSV tables. |

**Seeds are not optional here.** Run-to-run drift on this project has been the size of the effect — an arm that
scored 4 hits in 700 scored 2 on an identical re-run — so every headline is reported as mean ± sd over `SEEDS`.

In [ ]:
# The full study. Resumable: re-run this exact cell after any interruption.
# Add `loso` to STAGES for the 12-subject sweep (that stage alone is many hours x len(SEEDS)).
STAGES: str = 'audit encoder decoder ablation rebaseline analysis'
SEEDS: str = '42 43 44'  # >= 3, so every headline gets an honest error bar
HOLDOUT: str = 'ZAB'  # held-out subject for the single-fold stages

!SPATIAL=exact MEANING=keep \
    OUT_ROOT="{DRIVE_DIR}/experiments" \
    DRIVE_BACKUP="{DRIVE_DIR}/experiments" \
    DATA_CACHE="{PREPARED_LOCAL}" \
    ZTE_CACHE_REMOTE="{PREPARED_DRIVE}" \
    STAGES="{STAGES}" SEEDS="{SEEDS}" HOLDOUT="{HOLDOUT}" \
    bash scripts/run_zte_study.sh "{DATA_DIR}"
backup_prepared_cache()

## 7 · The proper benchmark — the live design choices, judged honestly

Two questions are now **settled** and their arms archived: CLIP against a frozen text encoder beat the
skip-gram/CBOW/masked/CPC sweep, and the raw conformer beat band power by 4x on the held-out board. What is still
open is *which part of the exp12 alignment stack earns its place* — so that is what this benchmark scores, on a
single fixed held-out subject (`ZAB`) so only the axis under test differs between rows, and on the **honest
held-out metric** (retrieval among the never-seen brain), never the pooled number that flatters every run:

- **the full stack vs its measured baseline** (`clip_e5_meaning_raw` → `zte_raw_aligned`),
- **each of the three changes in isolation** (alignment / adapter / orthogonality off),
- **encoder capacity once identity is handled** (`zte_raw_aligned` → `archive/zte_raw_aligned_wide`, which measured *worse*: 0.9523 against 0.9672),
- **calibration legitimacy** (`exp12_align_fit_train`: what the number becomes if a new user may never calibrate).

Watch **effective rank** alongside retrieval. Band power's collapse to eff-rank 0.160 is precisely what a
retrieval-only table hid last time; an arm that wins retrieval while shedding rank is buying it the same way.

Each arm is a full resumable `zte-run` (cached dataset, montage provisioned once, mirrored to Drive), so the
benchmark survives a runtime reset and re-running skips what is done. The table is sorted by held-out lift;
`zte-compare` renders the side-by-side HTML. To promote the winner to the full 12-subject trend, point
`scripts/run_loso.sh` at its config (Section 6a).

In [ ]:
import pandas as pd
import yaml

# The proper benchmark: the live A/B axes, each held out on ONE fixed subject so rows are comparable,
# each judged on the HONEST held-out scoreboard. Missing configs are skipped (Victor's in-flight arms).
BENCH_HOLDOUT = 'ZAB'
BENCH_OUT = f'{DRIVE_DIR}/benchmark' if 'DRIVE_DIR' in dir() else 'res/experiments/benchmark'
BENCH_CONFIGS = [
    ('experiments/flagship/clip_e5_meaning_raw.yaml', 'baseline: exp10 raw+meaning (32/700)'),
    ('experiments/flagship/zte_raw_aligned.yaml', 'exp12: full alignment stack'),
    ('experiments/ablation/exp12_align_off.yaml', 'exp12 - Euclidean alignment'),
    ('experiments/ablation/exp12_adapter_off.yaml', 'exp12 - subject adapter'),
    ('experiments/ablation/exp12_orthogonality_off.yaml', 'exp12 - identity orthogonality'),
    ('experiments/ablation/exp12_align_fit_train.yaml', 'exp12: no held-out calibration'),
]

# Train + evaluate each arm held out on ZAB (resumable, cached, Drive-backed).
for cfg, _ in BENCH_CONFIGS:
    if not pathlib.Path(cfg).exists():
        print('skip (config not present):', cfg)
        continue
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --spatial exact --meaning keep --data-cache "{PREPARED_LOCAL}" --loso-holdout {BENCH_HOLDOUT} --out-root "{BENCH_OUT}" --drive-backup "{BENCH_OUT}" --resume
backup_prepared_cache()

# ----- Tabulate the HONEST held-out scoreboard per arm (NOT the pooled sentence Top-1) -----
rows = []
for cfg, label in BENCH_CONFIGS:
    if not pathlib.Path(cfg).exists():
        continue
    rn = yaml.safe_load(open(cfg))['run_name']
    mp = pathlib.Path(f'{BENCH_OUT}/{rn}_lo{BENCH_HOLDOUT}/evaluation/metrics.json')
    if not mp.exists():
        print('no metrics yet for', rn)
        continue
    m = json.load(open(mp))
    sb = m.get('scoreboard', {}) or {}
    ho = sb.get('held_out_retrieval', {}) or {}
    cat = (m.get('honesty', {}).get('cross_subject_decode', {}).get('targets', {}) or {}).get('category', {}) or {}
    rows.append(
        {
            'config': label,
            'held_out_lift': ho.get('lift_top1'),
            'rank_pct': round(ho.get('rank_percentile', float('nan')), 4),
            'top5_hits': round((ho.get('top5') or 0) * (ho.get('n_queries') or 0)),
            'top5_p': ho.get('top5_p'),
            'eff_rank': round((m.get('embedding_health', {}) or {}).get('effective_rank_ratio', float('nan')), 3),
            'category': round(cat.get('mean', float('nan')), 3),
            'cat>chance': cat.get('above_chance'),
            'calib_lift': round(m.get('honesty', {}).get('calibration', {}).get('mean_lift', float('nan')), 3),
            'who_var': round(m.get('neurons', {}).get('who_variance', float('nan')), 3),
            'content_probe': (sb.get('content_probe') or {}).get('passes'),
        }
    )

# Render the side-by-side HTML, then show the honest table (sorted by held-out lift).
!uv run zte-compare --experiments "{BENCH_OUT}" --out "{BENCH_OUT}/COMPARE.html" --title "ZTE — proper benchmark (held out {BENCH_HOLDOUT})"
pd.DataFrame(rows).sort_values('held_out_lift', ascending=False) if rows else print('no completed arms yet')

In [ ]:
# Plot the benchmark: held-out retrieval lift (the honest headline) and category decode, per arm.
import matplotlib.pyplot as plt

if rows:
    bench = pd.DataFrame(rows).sort_values('held_out_lift')
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
    axes[0].barh(bench['config'], bench['held_out_lift'], color='#4c72b0')
    axes[0].axvline(0, color='k', lw=0.8)
    axes[0].set(title='held-out retrieval lift over chance (honest)', xlabel='lift')
    axes[1].barh(bench['config'], bench['category'], color='#55a868')
    axes[1].axvline(0.536, color='r', ls='--', lw=0.8, label='chance 0.536')
    axes[1].set(title='held-out category decode', xlabel='accuracy')
    axes[1].legend()
    for ax in axes:
        ax.grid(axis='x', alpha=0.3)
    fig.suptitle(f'ZTE proper benchmark — held out {BENCH_HOLDOUT} (seed 42)')
    plt.tight_layout()
    plt.show()
else:
    print('Run the benchmark cell above first.')

# Persist the benchmark to Drive alongside everything else (safe if you mounted Drive).
if 'backup_to_drive' in dir():
    backup_to_drive(note=f'proper benchmark — held out {BENCH_HOLDOUT} (seed 42)')

## 8 · The decoder — text out, on a 227k-parameter leash

The encoder is frozen. The language model (`Qwen/Qwen2.5-0.5B`) is frozen. The **only** trainable tensors in this section are the **226,560 parameters** of the prefix bridge between them — a LayerNorm, a rank-128 map, per-slot FiLM and a learned null prefix. That is the whole defence against the field's retracted EEG-to-text results: with the LM never updated there is no mechanism by which 700 ZuCo sentences can be memorised into the weights that produce text, so *"the output is corpus recall"* becomes a checkable claim rather than a matter of trust.

**Read this before you read any number this section prints.**

- **Sentence length is worth more bits than the encoder.** On the real 700-stimulus gallery $H(\text{identity}) = 9.4512$ bits and $H(\text{identity} \mid n_\text{words}) = 4.3090$, so **word count alone carries 5.1422 bits** of sentence identity. ZuCo's word segmentation comes from eye tracking, so `pad_mask` width *is* the word count and the model gets it for free. A length-only oracle at ±2 words scores Top-1 0.0214 / Top-5 0.0786 / Top-10 0.1371 against the best encoder's 0.0143 / 0.0457 / 0.0886 — it beats the encoder on every top-k. **8a measures that against your own checkpoint.**
- **The bit budget.** The encoder supplies ~4.7 bits of sentence identity; a 19.6-word English sentence needs ~190. That is **2.5%** of what free generation requires, so the honest expectation is a **null on generation** — written down in advance so the result cannot be graded on a moving target.
- **The powered readout is retrieval.** Decoder rescoring over the 700-sentence gallery is ~9.5 bits of forced choice at 700 queries, against a generation delta at *n* = 105. It is reported as **retrieval** everywhere it appears, never as generation.

**Prerequisite — the source encoder.** `decode_frozen_e5raw.yaml` mirrors `experiments/flagship/clip_e5_raw.yaml` (exp8) block-for-block, because the bridge reads the text space *that* run's `clip_head` was fitted to. Train `clip_e5_raw.yaml` held out on `ZAB` first — it is the last entry of the 5-iv series — and 8a/8b then find its `best.pt` on Drive automatically. `objective.text_source`, `text_backend` and `text_query_prefix` must match the source run exactly: rebuilding the frozen text target with a different encoder silently moves the space the inherited projection was fitted to.

**Never pass `--loso-holdout` to a decoder config.** It forces `split=by_subject_loso`, which shares all 700 texts between train and val — precisely the configuration in which a decoder recites the corpus and scores well — and the verdict then fails its `honest_split` clause. The decoder configs already name `loso_holdout_subject: ZAB` *inside* the honest four-cell split:

| Cell | Readings | Generalises over | Status |
| ---- | -------- | ---------------- | ------ |
| `train` | 11 x 525 = 5,775 | — | — |
| `val` — seen subject, unseen stimulus | 770 | language only | model selection |
| `test` — unseen subject, unseen stimulus | 105 | **both** | **the only headline cell** |
| `test_seen_stim` — unseen subject, seen stimulus | 525 | the brain only | diagnostic, labelled as one |

**The verdict gate.** `verdict.generation_above_controls` is an AND over five clauses, each reported with its numbers and an explicit `False` when it fails: an honest split, no candidate set, the paired bootstrap CI above zero against **every one** of the five brain-independent controls (`mean_prefix`, `null_prefix`, `phase`, `noise`, and a length-stratified `mismatch`), permutation *p* < 0.05, and mean prefix-influence KL ≥ 0.05 nats. A control that was requested but could not be decoded counts as **not beaten**, so losing one can never promote the verdict. **A generation number is never a headline unless that flag is `True`.**

> **The LM downloads on first use.** `Qwen/Qwen2.5-0.5B` (Apache-2.0, ~1 GB) comes from the Hub — Section 2's `HF_TOKEN` makes that authenticated and fast. Before any number leaves the repo, pin `decoder.lm_revision` to the commit SHA 8c prints: Hub repositories are mutable, and `null` resolves to whatever `main` happened to be that day.

Method, controls, the verdict gate and the pre-registered expectations in full: [`docs/DECODER.md`](../docs/DECODER.md).

### 8a · The length confound — read this before any decoder number

Trains nothing and gates nothing: it re-scores an existing checkpoint in minutes. Reports post-processing in
`{none, train-fitted, transductive}` x gallery in `{full 700, length-stratified}` against the length-only
oracle floor, plus the bit budget the decoder has to work inside.

In [ ]:
import pathlib

from IPython.display import Markdown, display  # type: ignore[import-untyped]

# Two source encoders, statistically tied on the 2026-08-13 held-out board: rank percentile 0.9635
# [0.9599, 0.9673] for exp8 against 0.9672 [0.9635, 0.9708] for exp12, overlapping intervals. The tie is
# not broken here; it is broken in 8d, on decoder rescoring. exp8 mirrors decode_frozen_e5raw, exp12
# mirrors decode_frozen_aligned.
ENCODER_RUN: str = 'exp8_clip_e5_raw_loZAB'
ALIGNED_ENCODER_RUN: str = 'exp12_zte_raw_aligned_loZAB'
DECODER_HOLDOUT: str = 'ZAB'  # fixed by the decoder configs' own train.loso_holdout_subject


def find_ckpt(run: str, which: str = 'best') -> str | None:
    """The first `<run>/checkpoints/<which>.pt` that exists — Drive before the local disk."""
    for root in (RUNS_DIR, f'{DRIVE_DIR}/benchmark', f'{DRIVE_DIR}/loso', LOCAL_RUNS):
        ckpt = pathlib.Path(f'{root}/{run}/checkpoints/{which}.pt')
        if ckpt.exists():
            return str(ckpt)
    return None


ENCODER_CKPT = find_ckpt(ENCODER_RUN)
ALIGNED_ENCODER_CKPT = find_ckpt(ALIGNED_ENCODER_RUN)
print('source encoder  ->', ENCODER_CKPT or f'MISSING — train {ENCODER_RUN} in Section 5-iv first')
print('aligned encoder ->', ALIGNED_ENCODER_CKPT or f'MISSING — train {ALIGNED_ENCODER_RUN} in 5-iv')

if ENCODER_CKPT:
    !uv run zte-rebaseline --ckpt "{ENCODER_CKPT}" --root "{DATA_DIR}" --holdout {DECODER_HOLDOUT} --length-tol 1 --oracle-tol 0,1,2,4
    _audit = pathlib.Path(ENCODER_CKPT).parent.parent / 'rebaseline' / 'rebaseline.md'
    if _audit.exists():
        display(Markdown(_audit.read_text(encoding='utf-8')))

### 8b · Train the prefix bridge

Neither the encoder nor the LM moves — only the bridge's 226,560 parameters. Stage 0 pretrains it on
`(text embedding -> text)` pairs with no EEG at all, leaving the EEG loop only the residual. With the encoder
frozen and in eval its output is a pure function of the reading, so a 25.8 MB embedding cache skips the raw
conformer after the warm-up.

> **No `--loso-holdout` here.** It would force `split=by_subject_loso` and fail the verdict's `honest_split`
> clause, which requires the `by_subject_and_stimulus` `test` cell.

In [ ]:
import yaml

DECODER_CFG: str = 'experiments/flagship/decode_zte_v2.yaml'  # the rebuilt headline arm
DECODER_RUNS: list[str] = [yaml.safe_load(open(DECODER_CFG))['run_name']]

if ENCODER_CKPT:
    !uv run zte-run --config {DECODER_CFG} --root "{DATA_DIR}" --spatial exact --data-cache "{PREPARED_LOCAL}" --encoder-ckpt "{ENCODER_CKPT}" --out-root "{DRIVE_DIR}/experiments" --resume
    backup_prepared_cache()  # persist the processed bundle to Drive so future sessions skip re-processing
else:
    print('No source encoder checkpoint — run Section 5-iv (clip_e5_raw.yaml, held out ZAB) first.')
print('decoder run ->', f'{DRIVE_DIR}/experiments/{DECODER_RUNS[0]}')

# (Offline wiring check — a 22,688-parameter LM built locally, synthetic data, nothing downloaded. It shows
# the three modes are wired; it is never a result. Uncomment to sanity-check the stack before a real run.)
# !uv run zte-run --config experiments/decoder/smoke/decode_tiny_mps.yaml --synthetic --mode encoder --name smoke_mps_encoder --out-root res/experiments
# !uv run zte-run --config experiments/decoder/smoke/decode_tiny_mps.yaml --synthetic --out-root res/experiments

### 8c · Read the decoder honestly

The verdict's five clauses first, then the two readouts in the right order: decoder-rescoring **retrieval**
(the powered one — ~9.5 bits of forced choice at 700 queries) and free-running **generation** (the expected
null at n = 105).

An absolute BLEU is not a result. The readable numbers are the paired delta against the **worst** control, the
permutation p, and the prefix-influence KL.

In [ ]:
import pandas as pd
from IPython.display import HTML, display  # type: ignore[import-untyped]


def decoder_run_dir(run: str) -> pathlib.Path | None:
    """An evaluated decoder run's directory — Drive before the local disk."""
    for root in (RUNS_DIR, LOCAL_RUNS):
        run_dir = pathlib.Path(f'{root}/{run}')
        if (run_dir / 'evaluation' / 'metrics.json').exists():
            return run_dir
    return None


def decoder_scorecard(runs: list[str]) -> pd.DataFrame:
    """One row per decoder arm: the verdict, the retrieval readout, then the generation delta."""
    out = []
    for run in runs:
        run_dir = decoder_run_dir(run)
        if run_dir is None:
            print('no evaluation yet for', run)
            continue
        m = json.load(open(run_dir / 'evaluation' / 'metrics.json'))
        verdict = m.get('verdict') or {}
        rescore = m.get('rescoring') or {}
        strat = rescore.get('length_stratified') or {}
        ci = verdict.get('generation_ci') or [None, None, None]
        out.append(
            {
                'run': run,
                'above_controls': verdict.get('generation_above_controls'),
                'cell': f'{verdict.get("generation_split_strategy")}/{verdict.get("generation_split_cell")}',
                'n': verdict.get('generation_n'),
                'rescore_top1': rescore.get('top1'),
                'rescore_chance': rescore.get('chance_top1'),
                'rescore_rank_pct': rescore.get('rank_percentile'),
                'strat_top1': strat.get('top1'),
                'strat_chance': strat.get('chance_top1'),
                'delta_vs_worst': ci[0],
                'delta_ci_lo': ci[1],
                'worst_control': verdict.get('generation_worst_control'),
                'perm_p': verdict.get('generation_permutation_p'),
                'prefix_kl': verdict.get('generation_prefix_kl'),
            }
        )
    return pd.DataFrame(out).set_index('run') if out else pd.DataFrame()


display(decoder_scorecard(DECODER_RUNS))

# The clause-by-clause breakdown, the frozen LM's identity, and the side-by-side page — for the headline
# arm. That page is the most persuasive artifact here: you can read the mean_prefix row saying almost the
# same thing as the hypothesis, which is what a paired delta of zero looks like in words.
_dir = decoder_run_dir(DECODER_RUNS[0])
if _dir is not None:
    _v = json.load(open(_dir / 'evaluation' / 'metrics.json')).get('verdict') or {}
    print('verdict clauses:', json.dumps(_v.get('generation_clauses'), indent=1))
    _gen = _dir / 'evaluation' / 'generation.json'
    if _gen.exists():
        _prov = json.load(open(_gen)).get('provenance') or {}
        print('frozen LM:', json.dumps(_prov.get('lm'), indent=1))
    _page = _dir / 'evaluation' / 'interactive' / 'generation.html'
    if _page.exists():
        display(HTML(_page.read_text(encoding='utf-8')))

# Pin decoder.lm_revision to this SHA before any number leaves the repo — Hub repositories are mutable.
!uv run python -c "from huggingface_hub import model_info; print('Qwen/Qwen2.5-0.5B @', model_info('Qwen/Qwen2.5-0.5B').sha)"

### 8d · The arms — the controls and ablations that make 8c readable

A decoder number on its own says nothing. These arms are what turn it into evidence. Each names the frozen source encoder it reuses, and each is `--resume`-safe, so re-running the cell below never redoes finished work.

| Arm | What it settles |
| --- | --- |
| `decode_frozen_aligned` | **The encoder tie-break.** Identical to the headline arm except for the encoder underneath it: `exp12_zte_raw_aligned` instead of `exp8_clip_e5_raw`. The two are tied on encoder retrieval (overlapping rank-percentile intervals), so the question moves to the readout that matters to a decoder — rescoring the 700-sentence gallery. Three dataset/model levers move with the encoder (`raw_align: euclidean`, `subject_signature`, `subject_adapter`); a frozen encoder handed features built under the other recipe silently sees a scale it never trained on, and nothing warns about it. |
| `decode_nostage0_ablation` | **Required reported ablation** — `stage0_epochs: 0`. Stage 0 teaches the bridge to emit fluent English from anything on the text manifold, so the EEG contribution is only a perturbation on a strong learned prior. If the delta vanishes without Stage 0, the honest reading is that Stage 0 was doing the work. |
| `decode_words_ablation` | **Registered ablation** — `conditioning: pooled_plus_words` adds 8 resampled word slots (762,496 more parameters) to the 8 pooled ones. It is an ablation and not the headline for two measured reasons: cross-subject word-level content is absent on ZuCo (`word_len` R² −0.0649, negative in 13/13 runs), and a length-L memory hands the decoder the word count, which is 5.14 bits of the answer. |
| `decode_joint_e5raw` | The encoder unfreezes after 3 stage-A epochs at a tenth of the bridge LR, with the CLIP loss kept on as an anchor so the decoding gradient cannot drag it out of the text space and orphan the bridge. The embedding cache is necessarily off here — the encoder moves. |
| `rebaseline_e5raw` | The length-confound audit arm: the *encoder* recipe trained on the **decoder's own** four-cell split, so 8a's floor is measured on the same partition the decoder is scored on rather than on a LOSO one. |
| `decode_encoder_only` | The regression control. `mode: encoder` must still reproduce `exp8_clip_e5_raw`'s history under the same seed — the proof that adding the decoder changed nothing upstream. Run it once; it is a full 40-epoch encoder run. |

**Pooling 12 folds.** The pre-registered primary generation analysis pools all 12 LOSO folds against one fixed stimulus partition (1,260 generations, fold-level bootstrap), because a single fold is only *n* = 105. Each fold needs its own source encoder (Section 6's sweep) and its own decoder config. Generate those **without touching the split**:

```sh
uv run zte-ablate generate --config experiments/decoder/decode_frozen_e5raw.yaml \
    --knob train.loso_holdout_subject --values ZAB,ZDM,ZGW,ZJM --out-dir res/decoder_folds
```

Then run each generated config with `--encoder-ckpt` pointing at that fold's encoder. Do **not** reach for `--loso-holdout`: it would replace the honest split with the one the verdict refuses to headline.

**Decoding a different cell.** Training already decodes `test`. `test_seen_stim` (unseen subject × *training* sentences) is a diagnostic, never a headline, and costs a full decode — five controls, the oracle and the rescoring pass — so it is opt-in:

```sh
uv run zte-decode --ckpt <run>/checkpoints/best.pt --root "$DATA_DIR" --split test_seen_stim \
    --out <run>/evaluation/test_seen_stim
```

### 8e · Run the decoder arms

Same `--resume` semantics as 5b. Every decoder arm reuses the **same** frozen source encoder, so nothing here
retrains it — the two `mode: encoder` arms train their own and are the expensive entries in the list.

In [ ]:
# One knob each against experiments/flagship/decode_zte_v2.yaml, so a win is attributable rather than asserted.
# Every arm reuses the SAME frozen source encoder -- nothing here retrains it.
DECODER_ARMS = [
    ('experiments/decoder/decode_v2_pooled.yaml', 'baseline: no ladder, no evidence (the v1 decoder)', ENCODER_CKPT),
    ('experiments/decoder/decode_v2_ladder_only.yaml', 'the rate ladder alone', ENCODER_CKPT),
    ('experiments/decoder/decode_v2_evidence_only.yaml', 'word-synchronous evidence alone', ENCODER_CKPT),
    (
        'experiments/decoder/decode_v2_no_length_stage.yaml',
        'required companion: the ladder without its reserved word-count stage',
        ENCODER_CKPT,
    ),
    ('experiments/decoder/decode_nostage0_ablation.yaml', 'required ablation: Stage 0 off', ENCODER_CKPT),
    ('experiments/decoder/rebaseline_e5raw.yaml', 'the encoder recipe on the decoder split', ENCODER_CKPT),
]

for cfg, why, ckpt in DECODER_ARMS:
    name = yaml.safe_load(open(cfg))['run_name']
    if not ckpt:
        print(f'skip {name}: no source encoder checkpoint yet ({why})')
        continue
    print(f'\n=== {name} — {why} ===')
    !uv run zte-run --config {cfg} --root "{DATA_DIR}" --spatial exact --data-cache "{PREPARED_LOCAL}" --encoder-ckpt "{ckpt}" --out-root "{DRIVE_DIR}/experiments" --resume
    DECODER_RUNS.append(name)
backup_prepared_cache()
print('\ndecoder runs ->', DECODER_RUNS)

## 9 · Visualise & interact (HTML)
Build the interactive **Thought-Space Explorer** + **Neuron Atlas** for a run, and the **comparison dashboard** across all runs. The dashboard is small enough to render inline; the 5 MB explorers are best downloaded (Section 10) and opened locally for full 3-D interaction.

In [ ]:
# Visualise a run — resolved from wherever it was written (Drive first, then the local disk).
_runs = run_dirs()
if _runs:
    _pick = next((d for d in _runs if 'zte_raw_aligned' in d.name), _runs[0])
    print('visualising', _pick)
    !uv run zte-visualize --run "{_pick}" --kind both
    !uv run zte-compare --experiments "{_pick.parent}" --out "{_pick.parent}/COMPARE.html"
    from IPython.display import HTML  # type: ignore[import-untyped]

    display(HTML(filename=f'{_pick.parent}/COMPARE.html'))
else:
    print('No runs found yet — train one in Section 5 first.')

In [ ]:
# The interactive held-out scoreboard: every "new brain" number as a gauge against its named reference.
# Written by every eval to <run>/evaluation/interactive/held_out_scoreboard.html.
import glob

from IPython.display import HTML  # type: ignore[import-untyped]

_boards = (
    sorted(glob.glob('res/experiments/*/evaluation/interactive/held_out_scoreboard.html'))
    + sorted(glob.glob(f'{DRIVE_DIR}/experiments/*/evaluation/interactive/held_out_scoreboard.html'))
    if 'DRIVE_DIR' in dir()
    else sorted(glob.glob('res/experiments/*/evaluation/interactive/held_out_scoreboard.html'))
)
if _boards:
    display(HTML(open(_boards[-1], encoding='utf-8').read()))  # newest run's dashboard, inline
else:
    print('No held-out scoreboard yet — train + evaluate a run (Section 5) first.')

## 9b · Explore the training you just ran — tables, charts & images
A quick, self-contained look at your runs **without leaving the notebook** — reads each run's `manifest.json` / `metrics.json` / figures directly (no ZTE import needed, so it works in the plain Colab kernel). You get a **scorecard DataFrame** across all runs, an interactive **run picker** (a Colab dropdown auto-populated from `res/experiments/`) that shows the selected run's **training curves** and key **evaluation figures** inline, a **comparison bar chart**, and a **metrics deep-dive** table for the selected run. (The PCA-by-subject thumbnail in the Section 9 dashboard now embeds itself as a data-URI, so it renders inline on Colab too.)

In [ ]:
# Every run's honest held-out headline in one table (reads evaluation/metrics.json). Pooled retrieval
# is deliberately absent: it includes the training subjects, so it is not evidence of generalisation.
import pandas as pd

rows = []
for d in run_dirs():
    mp = d / 'evaluation' / 'metrics.json'
    if not mp.exists():
        rows.append({'run': d.name, 'status': 'no evaluation yet'})
        continue
    m = json.load(open(mp))
    ho = (m.get('scoreboard', {}) or {}).get('held_out_retrieval', {}) or {}
    n_q = ho.get('n_queries') or 0
    ci = ho.get('rank_percentile_ci') or (None, None, None)
    rows.append(
        {
            'run': d.name,
            'status': 'ok',
            'rank_pct': ho.get('rank_percentile'),
            'ci_lo': ci[1],
            'ci_hi': ci[2],
            'top5_hits': round((ho.get('top5') or 0) * n_q),
            'top5_expected': round(5 * (ho.get('chance_top1') or 0) * n_q, 1),
            'top5_p': ho.get('top5_p'),
            'n_queries': n_q,
            'eff_rank': (m.get('embedding_health', {}) or {}).get('effective_rank_ratio'),
        }
    )
scorecard = pd.DataFrame(rows)
if 'rank_pct' in scorecard:
    scorecard = scorecard.sort_values('rank_pct', ascending=False, na_position='last')
scorecard.reset_index(drop=True)

In [ ]:
# Pick a run from a dropdown and view its training + evaluation figures (interactive, Colab widgets).
import glob
import os

from IPython.display import Image, Markdown, display  # type: ignore[import-untyped]

# Every run this notebook can see, wherever it was written (Drive first, then local).
RUN_PATHS = {d.name: d for d in run_dirs()}
RUNS = sorted(RUN_PATHS)
FIGURES = [
    ('Training curves (loss / lr)', 'checkpoints/training_curves.png'),
    # --- the blocking story: geometry & cross-subject retrieval ---
    (
        'Geometry before vs after (anti-cone / anti-hubness fix)',
        'evaluation/figures/geometry_before_after.png',
    ),
    (
        'Retrieval rank distribution (the pre-registered success metric)',
        'evaluation/figures/retrieval_rank_distribution.png',
    ),
    (
        'Cross-subject sentence retrieval (Top-K vs chance)',
        'evaluation/figures/retrieval_sentence.png',
    ),
    (
        'Cross-subject centroid similarity (hubness/identity diagnostic)',
        'evaluation/figures/subject_similarity.png',
    ),
    # --- what the space encodes: who vs what, and where on the scalp ---
    (
        'Variance budget — who (subject) vs what (word)',
        'evaluation/figures/variance_budget_pie.png',
    ),
    (
        'Neuron selectivity — top dimensions × attributes',
        'evaluation/figures/neuron_selectivity.png',
    ),
    (
        'Scalp topomap — electrodes carrying lexical-frequency info',
        'evaluation/figures/scalp_topomap.png',
    ),
    ('PCA of embeddings by subject', 'evaluation/figures/pca_by_subject.png'),
    ('Embedding health (per-dim std + PCA spectrum)', 'evaluation/figures/embedding_health.png'),
    (
        'Linear-probe comparison (ZTE vs raw vs noise vs phase-shuffled)',
        'evaluation/figures/probe_linear.png',
    ),
    ('Scalp-region importance heatmap', 'evaluation/figures/region_importance.png'),
]


def show_run_figures(run: str) -> None:
    """Render one run's figures inline (embeds the PNGs, so they show on Colab too)."""
    base = RUN_PATHS[run]
    display(Markdown(f'### `{run}` — training & evaluation figures'))
    for caption, rel in FIGURES:
        path = f'{base}/{rel}'
        if os.path.exists(path):
            display(Markdown(f'**{caption}** — `{rel}`'))
            display(Image(filename=path))
        else:
            display(Markdown(f'_missing: {rel}_'))


run_selector = None
if not RUNS:
    print('No runs yet — train one first (Section 5), then re-run this cell.')
else:
    try:
        import ipywidgets as widgets  # type: ignore[import-untyped]

        run_selector = widgets.Dropdown(options=RUNS, value=RUNS[0], description='Run:')
        # Reactive: changing the dropdown re-renders below; run_selector.value = current pick.
        widgets.interact(show_run_figures, run=run_selector)
    except ImportError:  # no widgets (e.g. plain terminal) -> just show the first run
        show_run_figures(RUNS[0])

In [ ]:
# Comparison bar chart across runs (matplotlib, from the scorecard above).
import matplotlib.pyplot as plt

comp = scorecard.dropna(subset=['rank_pct']).set_index('run') if 'rank_pct' in scorecard else None
if comp is not None and len(comp):
    ax = comp[['rank_pct', 'eff_rank']].plot.bar(figsize=(1.6 * len(comp) + 3, 4), rot=15)
    ax.set(title='Runs compared — held-out rank percentile & effective-rank ratio', ylabel='value')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='metric')
    plt.tight_layout()
    plt.show()
else:
    print('No evaluated runs yet — run a training cell first.')

In [ ]:
# Deep-dive: the full metrics.json for the SELECTED run (re-run after changing the dropdown above).
run = run_selector.value if run_selector is not None else (RUNS[0] if RUNS else None)
if run:
    metrics_path = f'{RUN_PATHS[run]}/evaluation/metrics.json'
    if os.path.exists(metrics_path):
        metrics = json.load(open(metrics_path))
        flat = pd.json_normalize(metrics, sep='.').T.rename(columns={0: 'value'})
        display(Markdown(f'### `{run}` — full evaluation metrics'))
        display(flat)
    else:
        print('No metrics.json — evaluation may have been skipped for', run)
else:
    print('No run available — train one first (Section 5).')

## 9c · Data analysis — the whole study, as charts you can interrogate

`zte-analyze` walks every run under one or more experiment trees and writes three things beside each other:

- **`ANALYSIS.html`** — one self-contained interactive page. Plotly is *inlined*, so it opens from a Drive
  mirror on a machine with no network. Sections: the honest headline, every fold and seed, what each lever is
  worth, the confounds, what the decoder wrote, the space itself, and training.
- **`tables/*.csv`** — every tidy frame behind the page (`runs`, `folds`, `multi_seed`, `feature_ablation`,
  `within_task`, `probes`, `subjects`, `generations`, `rebaseline`), so the analysis is redoable anywhere.
- **`ANALYSIS.md`** — the headline tables in prose, for a reader without a browser.

**How to read it.** `held_out_rank_percentile` first — it uses all 700 queries rather than only the ones that
landed. Top-1 at chance 1/700 expects exactly one hit, so 0.006 is three hits and noise. Then the confound
section: if the encoder sits below the length-only oracle curve, the Top-k is reproducing sentence length.
Then the control ladder: the warm bar has to clear **every** grey one, and `length_only` is the grey bar that
already knows the word count.

In [ ]:
# Collect every run -- this session's Drive folder first, then any earlier session and the local tree.
import glob
import os

ANALYSIS_DIR = f'{DRIVE_DIR}/analysis'
SOURCES = [f'{DRIVE_DIR}/experiments', *sorted(glob.glob(f'{ZTE_DRIVE}/*/experiments')), 'res/experiments']
SOURCES = [p for p in dict.fromkeys(SOURCES) if os.path.isdir(p)]
MONTAGE = 'res/montage_gsn105.csv' if os.path.exists('res/montage_gsn105.csv') else ''

print('reading:', *SOURCES, sep='\n  ')
!uv run zte-analyze --experiments {' '.join(f'"{s}"' for s in SOURCES)} --out "{ANALYSIS_DIR}" \
    --title "ZTE — study {RUN_DATE}" {f'--montage {MONTAGE}' if MONTAGE else ''}

In [ ]:
# The three tables that carry the argument, inline. Every cell is mean ± sd across seeds.
import math

import pandas as pd

pd.set_option('display.width', 200, 'display.max_columns', 60)


def _table(name: str) -> pd.DataFrame:
    path = f'{ANALYSIS_DIR}/tables/{name}.csv'
    return pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()


def _pm(frame: pd.DataFrame, metrics: list[str], keys: list[str]) -> pd.DataFrame:
    out = frame[[k for k in keys if k in frame]].copy()
    for m in metrics:
        if f'{m}_mean' in frame:
            pairs = zip(frame[f'{m}_mean'], frame[f'{m}_sd'], strict=True)
            # A one-seed arm has no spread; printing '± 0.0000' there would read as perfect stability.
            out[m] = [f'{a:.4f}' + (f' ± {b:.4f}' if math.isfinite(b) else '') for a, b in pairs]
    return out


HEADLINE = ['held_out_rank_percentile', 'held_out_top1', 'held_out_lift', 'stratified_top1', 'word_lift']
seeds, ablation, within = _table('multi_seed'), _table('feature_ablation'), _table('within_task')

if len(seeds):
    display(Markdown('### Headline over seeds — read `held_out_rank_percentile` first'))
    display(_pm(seeds, HEADLINE, ['arm', 'n_seeds', 'n_runs']))
if len(ablation):
    display(Markdown('### Feature ablation — one lever at a time'))
    display(_pm(ablation, HEADLINE[:4], ['question', 'level', 'n_runs']))
if len(within):
    display(Markdown('### Within-task pools — passage identity held fixed, so no passage shortcut is possible'))
    display(within.groupby(['arm', 'task'])[['top1', 'chance', 'lift', 'n_candidates']].mean().round(4))

In [ ]:
# A few panels drawn inline, so they are interactive in the notebook as well as in the page.
# The kernel is Colab's own interpreter and cannot import zte, so `zte-colab panels` draws them inside the venv
# and writes each one as Plotly figure JSON for this cell to render. Same list, same captions as the page.
import json

import plotly.io as pio
from IPython.display import Markdown, display  # type: ignore[import-untyped]

argv = ['uv', 'run', 'zte-colab', 'panels', '--experiments', *SOURCES, '--out', f'{ANALYSIS_DIR}/panels']
done = subprocess.run(
    [*argv, *(['--montage', MONTAGE] if MONTAGE else [])], capture_output=True, text=True, check=False
)
if done.returncode != 0:
    raise RuntimeError(f'`{" ".join(argv)}` failed:\n{done.stderr[-3000:]}')

PANELS = json.loads(done.stdout)
collected = PANELS['study']
print(f'{collected["runs"]} run(s) · {collected["folds"]} fold rows · {collected["generations"]} decode rows')
if collected['synthetic_runs']:
    print(f'WARNING: {collected["synthetic_runs"]} of them are SYNTHETIC and are wiring checks, not results.')

for panel in PANELS['panels']:
    display(Markdown(f'#### {panel["name"]} — {panel["caption"]}'))
    pio.read_json(panel['path']).show()

if PANELS['empty']:
    print('(skipped, no run carried the numbers for them: ' + ', '.join(PANELS['empty']) + ')')

In [ ]:
# The full page, embedded. It is a few MB with plotly inlined, so open it in a tab if the frame is slow:
# it is already on Drive at ANALYSIS_DIR/ANALYSIS.html and needs no network to render.
from IPython.display import IFrame

page = f'{ANALYSIS_DIR}/ANALYSIS.html'
print('page  ->', page)
print('md    ->', f'{ANALYSIS_DIR}/ANALYSIS.md')
print('csv   ->', f'{ANALYSIS_DIR}/tables/')
display(Markdown(open(f'{ANALYSIS_DIR}/ANALYSIS.md').read()) if os.path.exists(f'{ANALYSIS_DIR}/ANALYSIS.md') else '')

# Colab cannot iframe a Drive path directly, so serve a local copy of the page instead.
shutil.copy(page, 'ANALYSIS.html') if os.path.exists(page) else None
IFrame('ANALYSIS.html', width='100%', height=900) if os.path.exists('ANALYSIS.html') else 'Run 9c first.'

## 10 · Back up to Drive — lightweight archive + full snapshot
Two complementary saves, both landing under `Sharables/ZTE/{RUN_DATE}/` (permanent & shareable). **Smoke / `--synthetic` runs are skipped by default** — only real runs reach Drive (a session with only smoke runs is a friendly no-op; pass `include_synthetic=True` to force).

- **`backup_to_drive()`** — frequent, cheap. A provenance-stamped **best-only zip** (git commit + versions + each run's `config.yaml` + metrics) *and* a browsable mirror of reports/figures/3-D explorers. Call it after each **real** training step.
- **`snapshot_to_drive()`** — the **continue-locally** bundle. Zips the whole working state — `experiments` (real runs) + `cache` (the dataset cache!) + `benchmark` + `explorer` — into one file. Download it, `zte-pack unpack … --dest res` on your machine, and keep exploring / training **without paying for more GPU time**.

Both carry `PROVENANCE.json`/`PROVENANCE.md`. For long real-data runs, also write straight to Drive via `OUT_ROOT` (Sections 6/6b) so runs persist the instant they finish.

In [ ]:
!uv run zte-pack list

In [ ]:
# Back up EVERYTHING to Drive: all [best] runs as a provenance-stamped zip + a browsable mirror.
backup_to_drive(note=f'session {RUN_DATE}: all best runs')

# What you now have on Drive (permanent, shareable):
#   {DRIVE_DIR}/archives/zte_<date>_<time>.zip   restorable bundle (best.pt + config + eval + PROVENANCE.json/md)
#   {DRIVE_DIR}/experiments/<run>/               browsable reports, figures & 3-D explorers per run
#   {DRIVE_DIR}/benchmark/                        benchmark tables

# Restore later (new Colab session, or your locally) from the newest archive:
# !ls -t "{DRIVE_DIR}/archives"/*.zip | head -1
# !uv run zte-pack unpack "{DRIVE_DIR}/archives/<the>.zip" --dest res/experiments

# Download a single archive to your browser instead:
# from google.colab import files; files.download(f"{DRIVE_DIR}/archives/...zip")

# Free Colab space once it is safely on Drive (add --move to the zip, or):
# !uv run zte-pack clean experiments benchmark --yes

In [ ]:
# FULL snapshot -> Drive: experiments + cache + benchmark + explorer in ONE zip, so a local session
# can keep exploring or training with no GPU and no re-preparation.
snapshot_to_drive(note=f'full working-state snapshot {RUN_DATE}')

# Pick specific subtrees only (e.g. skip the big cache):
# snapshot_to_drive(targets=["experiments", "benchmark", "explorer"])

# Restore locally (or in a fresh Colab) — recreates res/experiments, res/cache, res/benchmark, res/explorer:
# !uv run zte-pack unpack "{DRIVE_DIR}/archives/<the_snapshot>.zip" --dest res

## 11 · Run it locally (inference)
Grab the newest archive from your Drive (`Sharables/ZTE/<date>/archives/`) — it carries `best.pt`, each run's `config.yaml`, the evaluation, and `PROVENANCE.json`/`PROVENANCE.md`. Then, in a terminal on your machine (Apple-silicon MPS is picked up automatically):

```sh
uv sync --group all
uv run zte-pack unpack ~/Downloads/zte_<date>_<time>.zip --dest res/experiments   # or straight from a synced Drive path
cat res/experiments/PROVENANCE.md            # git commit + versions + per-run metrics (how it was produced)

# Re-open the interactive explorer for a run (or evaluation/interactive/generation.html for a decoder run):
open res/experiments/exp12_zte_raw_aligned_loZAB/evaluation/interactive/thought_space_explorer.html

# Extract embeddings from the trained checkpoint (best.pt is enough — shapes + normaliser are baked in):
uv run zte-extract --ckpt res/experiments/exp12_zte_raw_aligned_loZAB/checkpoints/best.pt --root "/path/to/ZuCo Dataset" --out res/embeddings/exp12.npz

# Or re-run just the evaluation / comparison locally:
uv run zte-compare --experiments res/experiments
```

To reproduce a run exactly: `git checkout <commit from PROVENANCE.md>`, `uv sync --group all`, then `uv run zte-run --config res/experiments/<run>/config.yaml --root "/path/to/ZuCo Dataset" --name <run>`.

In [ ]:
%%bash
# 11b · Encode a brain the model has NEVER seen: a stranger supplies one short UNLABELLED baseline,
# whose channel covariance gives both the whitening map and the adapter signature. No ID, no retraining.
uv run python - <<'ZPY'
import os
from zte.config import ZTEConfig
from zte.data.dataset import ZuCoDataset
from zte.inference.embed import ZTEEmbedder

DATA_DIR, DRIVE_DIR = os.environ['DATA_DIR'], os.environ['DRIVE_DIR']
HOLDOUT = 'ZAB'
run_dir = f'{DRIVE_DIR}/experiments/exp12_zte_raw_aligned_lo{HOLDOUT}'
emb = ZTEEmbedder.from_checkpoint(f'{run_dir}/checkpoints/best.pt')

# Rebuild the exact feature pipeline the run used, UNALIGNED (the embedder applies its own maps):
dcfg = ZTEConfig.from_yaml(f'{run_dir}/config.yaml').dataset
dcfg.root, dcfg.raw_align = DATA_DIR, 'none'
ds = ZuCoDataset(dcfg).build(show_progress=False)
mask = (ds.words['subject'].to_numpy() == HOLDOUT) & ds.presence
raw = ds.raw_eeg[mask]                            # (n, n_channels, time_steps) unaligned windows
baseline, words = raw[:200], raw[200:]            # first ~200 words = the unlabelled calibration baseline

# One call registers BOTH their whitening map and their signature, from voltages alone.
emb.calibrate_subject(baseline_raw=baseline, subject_code=HOLDOUT)
vectors = emb.embed_signals(raw=words, subject_codes=[HOLDOUT] * len(words), show_progress=False)
print('encoded', vectors.shape, 'thought vectors for a calibrated new brain')
ZPY

## 12 · Housekeeping (free space / fresh clone)
Colab disk is small. Remove individual runs with **`remove_from_res('run_name')`** (defined in Section 4), or delete whole `res/` subtrees with `zte-pack clean`; or wipe the checkout and re-clone. In every case your **data and saved runs on Drive are untouched** — only local scratch is removed.

In [ ]:
# Remove specific runs locally (easy; does NOT touch Drive). Frees space after a smoke test:
remove_from_res('smoke_mps', 'smoke_mps_encoder')  # bare run names under res/experiments/
# remove_from_res('res/benchmark', 'res/cache')        # or any res/ subpath

# Or free whole res/ subtrees via the CLI (dry-run without --yes):
# !uv run zte-pack clean experiments cache benchmark --yes
# Wipe everything under res/:  !uv run zte-pack clean all --yes

In [ ]:
# Fresh clone — wipe the checkout and re-clone (e.g. after pushing critical updates).
# Your data + saved runs on Google Drive are NOT touched.
%cd /content
!rm -rf zte
!git clone --depth 1 https://github.com/victor-iyi/zte.git --branch main
%cd zte
!uv sync --group all
# Then re-run Section 2 (bootstrap) and re-mount Drive (Section 4).